# Cognitive E-Reader Colab Processor

This notebook does the heavy PDF processing outside the reader UI.

**Main output:** `cognitive_reader_bundle.json`

The TypeScript app only imports this JSON. It does not parse PDFs and does not run Python.

## Optional extras
- **Groq API enrichment:** add better summaries, goals, and definitions with a free Groq API key.
- **Colab TTS:** generate audio clips in Colab and embed them into the JSON bundle as data URIs.

No MediaPipe, WebGazer, webcam, or Readwise path is used.


In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install pymupdf pymupdf4llm requests gTTS
else:
    print('Not in Colab. Install pymupdf/pymupdf4llm/gTTS manually only if running locally.')


## Processor core

The following cell writes the tested helper functions into the runtime.


In [ ]:
from pathlib import Path
CORE_CODE = '"""Core helpers for the Cognitive E-Reader Colab processor.\n\nThese functions are intentionally small and mostly dependency-light so they can be tested\noutside Colab. The notebook imports or mirrors this logic, then uses optional PDF libraries\nfor extraction when running in Colab.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport time\nimport unicodedata\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nWORD_RE = re.compile(r"[A-Za-z][A-Za-z\'-]*")\nSENTENCE_RE = re.compile(r"(?<=[.!?])\\s+")\nCHAPTER_RE = re.compile(r"^(chapter\\s+\\d+|chapter\\s+[ivxlcdm]+|part\\s+\\d+|part\\s+[ivxlcdm]+)\\b", re.I)\n\nCOMMON_WORDS = {\n    "the", "and", "to", "of", "a", "in", "is", "it", "you", "that", "he", "was", "for", "on", "are",\n    "with", "as", "i", "his", "they", "be", "at", "one", "have", "this", "from", "or", "had", "by",\n    "not", "word", "but", "what", "some", "we", "can", "out", "other", "were", "all", "there", "when",\n    "up", "use", "your", "how", "said", "an", "each", "she", "which", "do", "their", "time", "if",\n    "will", "way", "about", "many", "then", "them", "would", "write", "like", "so", "these", "her",\n    "long", "make", "thing", "see", "him", "two", "has", "look", "more", "day", "could", "go", "come",\n}\n\n\n@dataclass(frozen=True)\nclass RawPage:\n    page: int\n    text: str\n\n\ndef now_iso() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef normalize_text(text: str) -> str:\n    text = unicodedata.normalize("NFKC", text)\n    text = text.replace("\\u00ad", "")\n    text = text.replace("\\r\\n", "\\n").replace("\\r", "\\n")\n    text = re.sub(r"[ \\t]+", " ", text)\n    text = re.sub(r"\\n{3,}", "\\n\\n", text)\n    return text.strip()\n\n\ndef repair_hyphenation(text: str) -> str:\n    return re.sub(r"([A-Za-z]{3,})-\\s*\\n\\s*([a-z]{3,})", r"\\1\\2", text)\n\n\ndef strip_repeated_edge_lines(pages: list[RawPage], threshold: float = 0.6) -> list[RawPage]:\n    if not pages:\n        return []\n    first_counts: dict[str, int] = {}\n    last_counts: dict[str, int] = {}\n    for page in pages:\n        lines = [line.strip() for line in page.text.splitlines() if line.strip()]\n        if not lines:\n            continue\n        first_counts[lines[0]] = first_counts.get(lines[0], 0) + 1\n        last_counts[lines[-1]] = last_counts.get(lines[-1], 0) + 1\n    cutoff = math.ceil(len(pages) * threshold)\n    repeated = {line for line, count in first_counts.items() if count >= cutoff}\n    repeated.update({line for line, count in last_counts.items() if count >= cutoff})\n\n    cleaned: list[RawPage] = []\n    for page in pages:\n        lines = [line for line in page.text.splitlines()]\n        while lines and lines[0].strip() in repeated:\n            lines.pop(0)\n        while lines and lines[-1].strip() in repeated:\n            lines.pop()\n        cleaned.append(RawPage(page=page.page, text="\\n".join(lines).strip()))\n    return cleaned\n\n\ndef split_paragraphs(text: str) -> list[str]:\n    text = normalize_text(repair_hyphenation(text))\n    chunks = re.split(r"\\n\\s*\\n", text)\n    paragraphs: list[str] = []\n    buffer: list[str] = []\n    for chunk in chunks:\n        chunk = re.sub(r"\\s+", " ", chunk).strip()\n        if not chunk:\n            continue\n        if len(chunk.split()) < 6 and not chunk.endswith((\'.\', \'!\', \'?\')):\n            if buffer:\n                paragraphs.append(" ".join(buffer).strip())\n                buffer = []\n            paragraphs.append(chunk)\n            continue\n        buffer.append(chunk)\n        if chunk.endswith((\'.\', \'!\', \'?\', \'”\', \'"\')):\n            paragraphs.append(" ".join(buffer).strip())\n            buffer = []\n    if buffer:\n        paragraphs.append(" ".join(buffer).strip())\n    return paragraphs\n\n\ndef word_count(text: str) -> int:\n    return len(WORD_RE.findall(text))\n\n\ndef sentence_count(text: str) -> int:\n    if not text.strip():\n        return 0\n    return max(1, len([part for part in SENTENCE_RE.split(text.strip()) if part.strip()]))\n\n\ndef estimate_syllables(word: str) -> int:\n    word = word.lower().strip("\'\\"")\n    if not word:\n        return 0\n    groups = re.findall(r"[aeiouy]+", word)\n    count = len(groups)\n    if word.endswith("e") and count > 1:\n        count -= 1\n    return max(1, count)\n\n\ndef rare_words(text: str, max_words: int = 6) -> list[str]:\n    seen: set[str] = set()\n    found: list[str] = []\n    for match in WORD_RE.finditer(text):\n        word = match.group(0).lower().strip("\'-")\n        if len(word) < 7 or word in COMMON_WORDS or word in seen:\n            continue\n        seen.add(word)\n        found.append(word)\n        if len(found) >= max_words:\n            break\n    return found\n\n\ndef difficulty_for_text(text: str) -> dict[str, Any]:\n    words = WORD_RE.findall(text)\n    wc = len(words)\n    sc = sentence_count(text)\n    syllables = sum(estimate_syllables(word) for word in words)\n    if wc == 0:\n        grade = 0.0\n    else:\n        grade = 0.39 * (wc / sc) + 11.8 * (syllables / wc) - 15.59\n    avg_sentence_length = wc / max(sc, 1)\n    rare_count = len(rare_words(text, max_words=50))\n    score = min(1.0, max(0.0, (grade / 16) * 0.6 + min(rare_count / 12, 1) * 0.4))\n    label = "easy" if score < 0.3 else "medium" if score <= 0.6 else "hard"\n    return {\n        "fleschKincaidGrade": round(grade, 2),\n        "avgSentenceLength": round(avg_sentence_length, 2),\n        "rareWordCount": rare_count,\n        "score": round(score, 3),\n        "label": label,\n    }\n\n\ndef stable_book_id(source_hash: str) -> str:\n    return f"book_{source_hash[:16]}"\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef infer_title(paragraphs: list[str], fallback: str = "Untitled Book") -> str:\n    for paragraph in paragraphs[:10]:\n        cleaned = paragraph.strip()\n        if 2 <= len(cleaned.split()) <= 14 and len(cleaned) <= 90:\n            return cleaned.title()\n    return fallback\n\n\ndef is_heading(text: str) -> bool:\n    stripped = text.strip()\n    if not stripped:\n        return False\n    if CHAPTER_RE.match(stripped):\n        return True\n    if len(stripped.split()) <= 9 and stripped == stripped.upper() and any(ch.isalpha() for ch in stripped):\n        return True\n    return False\n\n\ndef make_paragraph_id(chapter_index: int, section_index: int, paragraph_index: int) -> str:\n    return f"ch_{chapter_index:03d}.s_{section_index:03d}.p_{paragraph_index:03d}"\n\n\ndef make_section_id(chapter_index: int, section_index: int) -> str:\n    return f"ch_{chapter_index:03d}.s_{section_index:03d}"\n\n\ndef make_chapter_id(chapter_index: int) -> str:\n    return f"ch_{chapter_index:03d}"\n\n\ndef build_chapters(paragraphs: list[str]) -> list[dict[str, Any]]:\n    chapters: list[dict[str, Any]] = []\n    chapter_index = 0\n    current: dict[str, Any] | None = None\n    paragraph_order = 0\n\n    for text in paragraphs:\n        if current is None or is_heading(text):\n            chapter_index += 1\n            current = {\n                "chapterId": make_chapter_id(chapter_index),\n                "title": text if is_heading(text) else f"Chapter {chapter_index}",\n                "order": chapter_index,\n                "rawParagraphs": [],\n            }\n            chapters.append(current)\n            if is_heading(text):\n                continue\n        current["rawParagraphs"].append(text)\n\n    if not chapters:\n        chapters = [{"chapterId": "ch_001", "title": "Chapter 1", "order": 1, "rawParagraphs": paragraphs}]\n\n    finished: list[dict[str, Any]] = []\n    for chapter in chapters:\n        ch_num = chapter["order"]\n        section_id = make_section_id(ch_num, 1)\n        built_paragraphs: list[dict[str, Any]] = []\n        annotations: list[dict[str, Any]] = []\n        for idx, text in enumerate(chapter["rawParagraphs"], start=1):\n            paragraph_order += 1\n            pid = make_paragraph_id(ch_num, 1, idx)\n            diff = difficulty_for_text(text)\n            para = {\n                "id": pid,\n                "sectionId": section_id,\n                "order": paragraph_order,\n                "type": "paragraph",\n                "text": text,\n                "difficulty": diff,\n                "wordCount": word_count(text),\n                "sourceRefs": [],\n            }\n            built_paragraphs.append(para)\n            for word in rare_words(text, max_words=4):\n                match = re.search(rf"\\b{re.escape(word)}\\b", text, re.I)\n                if not match:\n                    continue\n                annotations.append({\n                    "id": f"vocab_{pid}_{word}",\n                    "type": "vocab",\n                    "target": {"paragraphId": pid, "startChar": match.start(), "endChar": match.end()},\n                    "payload": {\n                        "word": text[match.start():match.end()],\n                        "zipf": 3.0,\n                        "contextualDefinition": f"A possibly difficult word in this sentence: {word}.",\n                        "example": f"Try using {word} in your own sentence.",\n                    },\n                    "personaLevels": ["global_english", "young_reader"],\n                })\n        section_difficulty = average_difficulty([p["difficulty"] for p in built_paragraphs])\n        finished.append({\n            "chapterId": chapter["chapterId"],\n            "title": chapter["title"],\n            "order": chapter["order"],\n            "summary": f"This chapter contains {len(built_paragraphs)} processed paragraphs.",\n            "difficulty": section_difficulty,\n            "sections": [{\n                "id": section_id,\n                "title": chapter["title"],\n                "paragraphIds": [p["id"] for p in built_paragraphs],\n                "difficulty": section_difficulty,\n            }],\n            "paragraphs": built_paragraphs,\n            "annotations": annotations,\n            "missions": build_missions(chapter["chapterId"], built_paragraphs),\n            "lostCards": [],\n            "returnCards": [],\n            "narrationClips": [],\n        })\n    add_cards(finished)\n    return finished\n\n\ndef average_difficulty(items: Iterable[dict[str, Any]]) -> dict[str, Any]:\n    items = list(items)\n    if not items:\n        return difficulty_for_text("")\n    avg_score = sum(item["score"] for item in items) / len(items)\n    label = "easy" if avg_score < 0.3 else "medium" if avg_score <= 0.6 else "hard"\n    return {\n        "fleschKincaidGrade": round(sum(item["fleschKincaidGrade"] for item in items) / len(items), 2),\n        "avgSentenceLength": round(sum(item["avgSentenceLength"] for item in items) / len(items), 2),\n        "rareWordCount": int(round(sum(item["rareWordCount"] for item in items) / len(items))),\n        "score": round(avg_score, 3),\n        "label": label,\n    }\n\n\ndef build_missions(chapter_id: str, paragraphs: list[dict[str, Any]], target_words: int = 700) -> list[dict[str, Any]]:\n    missions: list[dict[str, Any]] = []\n    bucket: list[dict[str, Any]] = []\n    bucket_words = 0\n    mission_index = 1\n\n    def flush() -> None:\n        nonlocal bucket, bucket_words, mission_index\n        if not bucket:\n            return\n        diff = average_difficulty([p["difficulty"] for p in bucket])\n        mid = f"{chapter_id}_m_{mission_index:03d}"\n        missions.append({\n            "id": mid,\n            "chapterId": chapter_id,\n            "paragraphIds": [p["id"] for p in bucket],\n            "wordCount": bucket_words,\n            "timeEstimateMinutes": max(1, round(bucket_words / 180)),\n            "difficulty": diff["label"],\n            "goal": "Read this section and leave with one clear idea.",\n            "exitPrompt": "What is the most important idea in this section?",\n            "warmupTerms": [],\n        })\n        mission_index += 1\n        bucket = []\n        bucket_words = 0\n\n    for paragraph in paragraphs:\n        bucket.append(paragraph)\n        bucket_words += paragraph["wordCount"]\n        if bucket_words >= target_words:\n            flush()\n    flush()\n    return missions\n\n\ndef add_cards(chapters: list[dict[str, Any]]) -> None:\n    for chapter in chapters:\n        for mission in chapter["missions"]:\n            chapter["lostCards"].append({\n                "id": f"lost_{mission[\'id\']}",\n                "targetMissionId": mission["id"],\n                "mainIdea": "This section is part of the processed book structure.",\n                "simplerVersion": "Read it slowly and identify the one main point.",\n                "importantTerms": [],\n                "rereadParagraphIds": mission["paragraphIds"][:1],\n                "rereadReason": "The first paragraph usually sets up the section.",\n                "thinkingQuestion": "What is this section trying to make you understand?",\n            })\n            chapter["returnCards"].append({\n                "id": f"return_{mission[\'id\']}",\n                "targetMissionId": mission["id"],\n                "lastSummary": "You were reading this processed section.",\n                "nextPreview": "Continue from the next paragraph and rebuild the context.",\n                "openQuestion": "What do you remember from this mission?",\n            })\n\n\ndef build_difficulty_terrain(chapters: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    terrain: list[dict[str, Any]] = []\n    for chapter in chapters:\n        for paragraph in chapter["paragraphs"]:\n            terrain.append({\n                "paragraphId": paragraph["id"],\n                "order": paragraph["order"],\n                "score": paragraph["difficulty"]["score"],\n                "label": paragraph["difficulty"]["label"],\n                "wordCount": paragraph["wordCount"],\n            })\n    return terrain\n\n\ndef build_bundle_from_text(text: str, title: str | None = None, source_hash: str | None = None) -> dict[str, Any]:\n    start = time.perf_counter()\n    paragraphs = split_paragraphs(text)\n    source_hash = source_hash or hashlib.sha256(text.encode("utf-8")).hexdigest()\n    chapters = build_chapters(paragraphs)\n    terrain = build_difficulty_terrain(chapters)\n    total_words = sum(p["wordCount"] for ch in chapters for p in ch["paragraphs"])\n    all_diffs = [p["difficulty"] for ch in chapters for p in ch["paragraphs"]]\n    book_diff = average_difficulty(all_diffs)\n    inferred_title = title or infer_title(paragraphs)\n    book_id = stable_book_id(source_hash)\n\n    manifest = {\n        "schemaVersion": "1.0.0",\n        "bookId": book_id,\n        "sourceHash": source_hash,\n        "title": inferred_title,\n        "author": None,\n        "language": "en",\n        "bookType": "unknown",\n        "bookDna": {\n            "mainTheme": "Processed reading bundle",\n            "coreArgument": "This book was converted into clean paragraphs, annotations, and missions.",\n            "totalReadingTimeMinutes": max(1, round(total_words / 180)),\n            "difficultyLabel": book_diff["label"],\n            "emotionalTone": "unknown",\n            "keyConcepts": rare_words(text, max_words=8),\n            "bookType": "unknown",\n            "recommendedSessionLengthMinutes": 15,\n        },\n        "createdAt": now_iso(),\n        "capabilities": {\n            "vocabulary": True,\n            "lostCards": True,\n            "missions": True,\n            "cleanReturn": True,\n            "search": False,\n            "audio": False,\n            "embeddedTts": False,\n            "difficultyTerrain": True,\n        },\n        "readerDefaults": {"theme": "warm", "fontSize": 18, "lineHeight": 1.75, "maxWidthCh": 65},\n        "chapters": [{\n            "chapterId": ch["chapterId"],\n            "order": ch["order"],\n            "title": ch["title"],\n            "summary": ch["summary"],\n            "difficulty": ch["difficulty"],\n            "missionCount": len(ch["missions"]),\n            "paragraphCount": len(ch["paragraphs"]),\n            "wordCount": sum(p["wordCount"] for p in ch["paragraphs"]),\n        } for ch in chapters],\n        "difficultyTerrain": terrain,\n        "report": {\n            "pagesTotal": 0,\n            "pagesProcessed": 0,\n            "extractionMethod": "text_input",\n            "ocrUsed": False,\n            "chaptersDetected": len(chapters),\n            "paragraphsDetected": sum(len(ch["paragraphs"]) for ch in chapters),\n            "imagesExtracted": 0,\n            "warnings": [],\n            "processingTimeSeconds": round(time.perf_counter() - start, 3),\n        },\n    }\n    return {\n        "schemaVersion": "1.0.0",\n        "generatedAt": now_iso(),\n        "source": "colab_notebook",\n        "manifest": manifest,\n        "chapters": chapters,\n    }\n\n\ndef write_bundle(bundle: dict[str, Any], path: str | Path) -> None:\n    Path(path).write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef audio_bytes_to_data_uri(audio_bytes: bytes, mime_type: str = "audio/mpeg") -> str:\n    import base64\n    encoded = base64.b64encode(audio_bytes).decode("ascii")\n    return f"data:{mime_type};base64,{encoded}"\n\n\ndef make_narration_clip(\n    clip_id: str,\n    chapter_id: str,\n    paragraph_ids: list[str],\n    audio_bytes: bytes,\n    mime_type: str = "audio/mpeg",\n    voice: str | None = None,\n    duration_seconds: float | None = None,\n) -> dict[str, Any]:\n    return {\n        "id": clip_id,\n        "chapterId": chapter_id,\n        "paragraphIds": paragraph_ids,\n        "mimeType": mime_type,\n        "audioDataUri": audio_bytes_to_data_uri(audio_bytes, mime_type),\n        "voice": voice,\n        "durationSeconds": duration_seconds,\n        "alignment": [],\n        "generatedBy": "colab_tts",\n    }\n\n\ndef attach_narration_clip(bundle: dict[str, Any], chapter_id: str, clip: dict[str, Any]) -> bool:\n    for chapter in bundle.get("chapters", []):\n        if chapter.get("chapterId") == chapter_id:\n            chapter.setdefault("narrationClips", []).append(clip)\n            bundle["manifest"]["capabilities"]["audio"] = True\n            bundle["manifest"]["capabilities"]["embeddedTts"] = True\n            return True\n    return False\n'
Path('/content/processor_core.py').write_text(CORE_CODE, encoding='utf-8')
print('processor_core.py written')


In [ ]:
import sys
sys.path.insert(0, '/content')
from processor_core import *
print('Core imported. Running notebook helper self-tests...')

assert normalize_text('A\r\n  B') == 'A\n B'
assert repair_hyphenation('psycho-\nlogical') == 'psychological'
assert word_count('One two three.') == 3
assert sentence_count('One. Two?') == 2
assert make_paragraph_id(1, 2, 3) == 'ch_001.s_002.p_003'
assert build_bundle_from_text('Chapter 1\n\nThis is a contextual annotation test.')['schemaVersion'] == '1.0.0'
assert make_narration_clip('c1', 'ch_001', ['p1'], b'abc')['audioDataUri'].startswith('data:audio/mpeg;base64,')
print('Self-tests passed.')


## Upload PDF

Run this cell in Colab and choose your PDF.


In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = Path(next(iter(uploaded.keys())))
else:
    PDF_PATH = Path('input.pdf')

print('PDF path:', PDF_PATH)


## Extract text from PDF

Primary: `pymupdf4llm`. Fallback: raw `PyMuPDF`.


In [ ]:
def extract_pdf_pages(pdf_path):
    pdf_path = str(pdf_path)
    pages = []
    method = 'unknown'
    warnings = []

    try:
        import pymupdf4llm
        chunks = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
        for idx, chunk in enumerate(chunks, start=1):
            text = chunk.get('text', '') if isinstance(chunk, dict) else str(chunk)
            pages.append(RawPage(page=idx, text=normalize_text(text)))
        method = 'pymupdf4llm'
    except Exception as exc:
        warnings.append(f'pymupdf4llm failed: {exc}')
        try:
            import fitz
            doc = fitz.open(pdf_path)
            for idx, page in enumerate(doc, start=1):
                pages.append(RawPage(page=idx, text=normalize_text(page.get_text('text'))))
            method = 'pymupdf_fallback'
        except Exception as fallback_exc:
            raise RuntimeError(f'PDF extraction failed: {fallback_exc}') from fallback_exc

    pages = strip_repeated_edge_lines(pages)
    pages = [RawPage(page=p.page, text=repair_hyphenation(p.text)) for p in pages if p.text.strip()]
    return pages, method, warnings

pages, extraction_method, extraction_warnings = extract_pdf_pages(PDF_PATH)
print('Extraction method:', extraction_method)
print('Pages extracted:', len(pages))
print('Warnings:', extraction_warnings)
print('Preview:', pages[0].text[:500] if pages else 'NO TEXT')


## Build the reader bundle


In [ ]:
import time

start = time.perf_counter()
full_text = '\n\n'.join(page.text for page in pages)
source_hash = sha256_file(PDF_PATH)
bundle = build_bundle_from_text(full_text, source_hash=source_hash)

bundle['manifest']['report']['pagesTotal'] = len(pages)
bundle['manifest']['report']['pagesProcessed'] = len(pages)
bundle['manifest']['report']['extractionMethod'] = extraction_method
bundle['manifest']['report']['warnings'] = extraction_warnings
bundle['manifest']['report']['processingTimeSeconds'] = round(time.perf_counter() - start, 3)

print('Book title:', bundle['manifest']['title'])
print('Book id:', bundle['manifest']['bookId'])
print('Chapters:', len(bundle['chapters']))
print('Paragraphs:', bundle['manifest']['report']['paragraphsDetected'])


## Optional: Groq API enrichment

Use this if you want better summaries/missions/Book DNA.

1. Get a Groq API key.
2. Paste it when prompted.
3. This updates the bundle before export.

This is optional; the reader works without it.


In [ ]:
USE_GROQ = False  # Change to True in Colab when you want LLM enrichment
GROQ_MODEL = 'llama-3.1-8b-instant'

def groq_chat_json(api_key, prompt, model=GROQ_MODEL):
    import json, requests
    response = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={
            'model': model,
            'messages': [
                {'role': 'system', 'content': 'Return only compact valid JSON. No markdown.'},
                {'role': 'user', 'content': prompt},
            ],
            'temperature': 0.2,
            'max_tokens': 900,
        },
        timeout=60,
    )
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    return json.loads(content)

def enrich_bundle_with_groq(bundle, api_key):
    sample_text = '\n\n'.join(
        p['text'] for ch in bundle['chapters'][:2] for p in ch['paragraphs'][:8]
    )[:7000]
    prompt = f'''Analyze this book sample and return JSON with keys:
mainTheme, coreArgument, emotionalTone, keyConcepts array, bookType, recommendedSessionLengthMinutes, chapterSummary, missionGoal, exitPrompt.

TEXT:
{sample_text}
'''
    data = groq_chat_json(api_key, prompt)
    dna = bundle['manifest']['bookDna']
    dna['mainTheme'] = data.get('mainTheme', dna['mainTheme'])
    dna['coreArgument'] = data.get('coreArgument', dna['coreArgument'])
    dna['emotionalTone'] = data.get('emotionalTone', dna['emotionalTone'])
    dna['keyConcepts'] = data.get('keyConcepts', dna['keyConcepts'])[:10]
    dna['recommendedSessionLengthMinutes'] = int(data.get('recommendedSessionLengthMinutes', dna['recommendedSessionLengthMinutes']))
    if data.get('bookType'):
        bundle['manifest']['bookType'] = data['bookType']
        dna['bookType'] = data['bookType']
    for chapter in bundle['chapters']:
        chapter['summary'] = data.get('chapterSummary', chapter['summary'])
        for mission in chapter['missions']:
            mission['goal'] = data.get('missionGoal', mission['goal'])
            mission['exitPrompt'] = data.get('exitPrompt', mission['exitPrompt'])
    bundle['manifest']['report']['warnings'].append('Groq enrichment applied.')
    return bundle

if USE_GROQ:
    import getpass
    GROQ_API_KEY = getpass.getpass('Paste GROQ_API_KEY: ')
    bundle = enrich_bundle_with_groq(bundle, GROQ_API_KEY)
    print('Groq enrichment complete.')
else:
    print('Groq enrichment skipped. Set USE_GROQ=True to enable.')


## Optional: generate embedded TTS clips in Colab

This creates small MP3 clips and embeds them directly into the JSON.

Important: embedded audio makes the JSON large. Start with 1-3 chapters or a few clips.


In [ ]:
GENERATE_TTS = False  # Change to True to embed TTS clips
MAX_TTS_CHAPTERS = 1
MAX_WORDS_PER_CLIP = 650

def make_tts_text_for_chapter(chapter, max_words=MAX_WORDS_PER_CLIP):
    words = []
    paragraph_ids = []
    for p in chapter['paragraphs']:
        paragraph_ids.append(p['id'])
        words.extend(p['text'].split())
        if len(words) >= max_words:
            break
    return ' '.join(words[:max_words]), paragraph_ids

if GENERATE_TTS:
    from gtts import gTTS
    import tempfile
    for chapter in bundle['chapters'][:MAX_TTS_CHAPTERS]:
        text, paragraph_ids = make_tts_text_for_chapter(chapter)
        if not text.strip():
            continue
        tts = gTTS(text=text, lang=bundle['manifest'].get('language', 'en'))
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
            tts.save(tmp.name)
            audio_bytes = Path(tmp.name).read_bytes()
        clip = make_narration_clip(
            clip_id=f"tts_{chapter['chapterId']}_001",
            chapter_id=chapter['chapterId'],
            paragraph_ids=paragraph_ids,
            audio_bytes=audio_bytes,
            mime_type='audio/mpeg',
            voice='gTTS',
        )
        attach_narration_clip(bundle, chapter['chapterId'], clip)
        print('Attached TTS clip:', clip['id'], 'size chars:', len(clip['audioDataUri']))
else:
    print('TTS generation skipped. Set GENERATE_TTS=True to enable.')


## Validate bundle shape


In [ ]:
def validate_export_bundle(bundle):
    assert bundle['schemaVersion'] == '1.0.0'
    assert bundle['manifest']['schemaVersion'] == '1.0.0'
    assert isinstance(bundle['chapters'], list) and bundle['chapters']
    paragraph_ids = set()
    for chapter in bundle['chapters']:
        assert chapter['chapterId']
        for paragraph in chapter['paragraphs']:
            assert paragraph['id'] not in paragraph_ids
            paragraph_ids.add(paragraph['id'])
            assert paragraph['text'].strip()
            assert 0 <= paragraph['difficulty']['score'] <= 1
        for annotation in chapter['annotations']:
            assert annotation['target']['paragraphId'] in paragraph_ids
            assert annotation['target']['endChar'] > annotation['target']['startChar']
        for clip in chapter.get('narrationClips', []):
            assert clip['audioDataUri'].startswith('data:audio/')
            assert clip['generatedBy'] == 'colab_tts'
    assert len(bundle['manifest']['difficultyTerrain']) == len(paragraph_ids)
    return True

validate_export_bundle(bundle)
print('Bundle validation passed.')


## Export JSON

Download this file and import it into the Cognitive E-Reader UI.


In [ ]:
OUTPUT_PATH = Path('/content/cognitive_reader_bundle.json') if IN_COLAB else Path('cognitive_reader_bundle.json')
write_bundle(bundle, OUTPUT_PATH)
print('Wrote:', OUTPUT_PATH)

if IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
